# Day 1 check on Kaggle: offline guideline search and R1 counter

This notebook shows that the guideline search (RAG v1) and the R1 building counter run on Kaggle
**with the internet turned off and nothing installed**.

**Before you run (right panel, Session options):**
- Accelerator: **None**. CPU is enough, so no GPU quota is used.
- Internet: **Off**.
- Input: **+ Add Input** and add your `offline-rag-kr2` dataset.

Then **Run All**. It takes about a minute.

## 1. Find the dataset

In [1]:
import glob, os, sys, zipfile

os.environ["HF_HUB_OFFLINE"] = "1"  # never try to reach Hugging Face


def find_rag_dir():
    hits = glob.glob("/kaggle/input/**/guideline_index.py", recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    # If Kaggle kept the uploaded zip as a zip, unpack it into /kaggle/working
    zips = glob.glob("/kaggle/input/**/offline-rag-kr2*.zip", recursive=True)
    if zips:
        with zipfile.ZipFile(zips[0]) as z:
            z.extractall("/kaggle/working/offline-rag-kr2")
        hits = glob.glob("/kaggle/working/offline-rag-kr2/**/guideline_index.py", recursive=True)
        if hits:
            return os.path.dirname(hits[0])
    raise FileNotFoundError("Dataset not found. Add the offline-rag-kr2 dataset with + Add Input.")


RAG_DIR = find_rag_dir()
sys.path.insert(0, RAG_DIR)
print("Dataset folder:", RAG_DIR)
print(sorted(os.listdir(RAG_DIR)))

Dataset folder: /kaggle/input/datasets/riditaafrin/offline-rag-kr2
['README.md', 'guideline_chunks.jsonl', 'guideline_embeddings.npy', 'guideline_index.py', 'hybrid_bdc_counter.py', 'models', 'sanity_queries.json']


## 2. Check that the internet is off

In [2]:
import socket

try:
    socket.create_connection(("huggingface.co", 443), timeout=5).close()
    INTERNET = "on"
    print("Internet is ON. For the offline proof, turn it off in Session options and run again.")
except OSError:
    INTERNET = "off"
    print("Internet is OFF: nothing below can download anything.")

Internet is OFF: nothing below can download anything.


## 3. Search the guidelines
Kaggle has no `chromadb`, so the search uses the saved vectors (`guideline_embeddings.npy`)
and embeds the question with the same model through `transformers` + `torch`.

In [3]:
import importlib.util
import guideline_index as gi

print("chromadb installed:", importlib.util.find_spec("chromadb") is not None)
print("Search backend:", gi.search_backend())
print()
print(gi.format_results(gi.search_guidelines("building completely collapsed, total collapse", k=3)))

chromadb installed: False


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Search backend: numpy + torch

[EMS98-rc-frame-mexico-city-2] (EMS-98, pp. 82-83)
RC frame Mexico City 1985
GRADE OF DAMAGE
Figure 5 - 19
TYPE OF STRUCTURE EARTHQUAKE / SITE
M
Comment:
This building has suffered partial collapse of the upper part. Although single upper storeys have failed, no part of the building has collapsed completely to ground level, so the damage is only grade 4.

[EMS98-rc-frame-north-pelopponissos-2] (EMS-98, pp. 81-82)
RC frame North Pelopponissos, Greece 1995 / Aegion
GRADE OF DAMAGE
Figure 5 - 18
TYPE OF STRUCTURE EARTHQUAKE / SITE
M
Comment:
The middle part of this building has collapsed completely, making the damage of grade 5.

[INSARAG2B-5.8.2-p1] (INSARAG Guidelines 2020, Vol. II Manual B Operations, pp. 37-38)
5.8 Worksite Triage > 5.8.2 Second order of priority: Building information and operational constraints
In cases where the UCC needs to use additional information to list the worksites in order of priority, building related and operations related i

## 4. Same answers as on the laptop?

In [4]:
import json

with open(os.path.join(RAG_DIR, "sanity_queries.json"), encoding="utf-8") as f:
    expected = json.load(f)

same = 0
for item in expected:
    got = [r["id"] for r in gi.search_guidelines(item["query"], k=3)]
    ok = got == item["top3"]
    same += ok
    print("SAME" if ok else "DIFF", "|", item["query"][:60])
    if not ok:
        print("     laptop:", item["top3"])
        print("     kaggle:", got)
print()
print(f"{same}/{len(expected)} questions give the same top 3 as on the laptop.")

SAME | what does destroyed mean in the xBD damage labels
SAME | minor damage roof elements missing visible cracks
SAME | building completely collapsed, scorched or no longer present
SAME | how to classify building damage from satellite imagery
SAME | possibly damaged class when image quality is poor
SAME | how to tell the damage grade of a masonry house
SAME | reinforced concrete frame building with collapsed ground flo
SAME | most buildings destroyed in the area, very destructive earth
SAME | search and rescue priority for collapsed buildings
SAME | assessment search and rescue levels ASR
SAME | worksite triage of collapsed structures
SAME | how much covered living space per person in a shelter
SAME | emergency shelter for displaced families after a disaster
SAME | household items to distribute after disaster
SAME | reconstruction and long-term recovery of housing
SAME | 7 destroyed, 12 damaged buildings; search and rescue priorit

16/16 questions give the same top 3 as on the laptop.

## 5. R1 building counter

In [5]:
import numpy as np
import hybrid_bdc_counter as hb

# A made-up R1 mask: 3 intact, 2 damaged and 1 destroyed building (labels 1, 2, 3)
mask = np.zeros((1024, 1024), np.uint8)
for x, label in [(50, 1), (200, 1), (350, 1), (500, 2), (650, 2), (800, 3)]:
    mask[100:160, x:x + 60] = label

counts = hb.count_by_severity(mask, mode="r1")
print(counts)
assert (counts["intact"], counts["damaged"], counts["destroyed"]) == (3, 2, 1), "Counter gave wrong counts"
print("Counter OK: 3 intact, 2 damaged, 1 destroyed.")

{'intact': 3, 'damaged': 2, 'destroyed': 1, 'total_damaged': 3}
Counter OK: 3 intact, 2 damaged, 1 destroyed.


## 6. Save the result

In [6]:
summary = {
    "internet": INTERNET,
    "search_backend": gi.search_backend(),
    "same_top3_as_laptop": f"{same}/{len(expected)}",
    "counter_test": counts,
}
with open("/kaggle/working/day1_kaggle_check.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

{
  "internet": "off",
  "search_backend": "numpy + torch",
  "same_top3_as_laptop": "16/16",
  "counter_test": {
    "intact": 3,
    "damaged": 2,
    "destroyed": 1,
    "total_damaged": 3
  }
}


**What you should see:** `Internet is OFF`, `Search backend: numpy + torch`, `16/16`, and `Counter OK`.
Then click **Save Version** and choose **Save & Run All (Commit)**, so Kaggle runs the whole notebook again with these settings and keeps the outputs as proof. Never edit an output by hand.